In [1]:
import os
import pandas as pd

print("Input folders:", os.listdir("../input"))

# ---------- Load training data ----------
train_path = "../input/champs-scalar-coupling/train.csv"
train_df = pd.read_csv(train_path)

required_cols = {
    "type",
    "scalar_coupling_constant",
    "molecule_name",
    "atom_index_0",
    "atom_index_1",
}
assert required_cols.issubset(
    train_df.columns
), "Training file missing required columns."

# ---------- Per‑type mean ----------
type_means = train_df.groupby("type")["scalar_coupling_constant"].mean()
print("Computed mean values for each type:")
print(type_means)

# ---------- Load auxiliary data ----------
structures_path = "../input/champs-scalar-coupling/structures.csv"
structures_df = pd.read_csv(structures_path)
structures_df = structures_df.rename(columns={"atom": "element"})

contrib_path = "../input/champs-scalar-coupling/scalar_coupling_contributions.csv"
contrib_df = pd.read_csv(contrib_path)
contrib_df["contrib_sum"] = (
    contrib_df["fc"] + contrib_df["sd"] + contrib_df["pso"] + contrib_df["dso"]
)

# ---------- Merge atom symbols into training ----------
train_merged = train_df.merge(
    structures_df[["molecule_name", "atom_index", "element"]].rename(
        columns={"atom_index": "atom_index_0", "element": "atom0"}
    ),
    on=["molecule_name", "atom_index_0"],
    how="left",
)
train_merged = train_merged.merge(
    structures_df[["molecule_name", "atom_index", "element"]].rename(
        columns={"atom_index": "atom_index_1", "element": "atom1"}
    ),
    on=["molecule_name", "atom_index_1"],
    how="left",
)

assert (
    not train_merged[["atom0", "atom1"]].isnull().any().any()
), "Missing atom symbols in training merge."

alpha = 10.0  # Bayesian smoothing strength

# ---------- Refined (shrunken) means ----------
type_atom_stats = (
    train_merged.groupby(["type", "atom0", "atom1"])["scalar_coupling_constant"]
    .agg(["mean", "count"])
    .reset_index()
)
type_atom_stats["type_mean"] = type_atom_stats["type"].map(type_means)
type_atom_stats["refined_mean"] = (
    type_atom_stats["mean"] * type_atom_stats["count"]
    + alpha * type_atom_stats["type_mean"]
) / (type_atom_stats["count"] + alpha)
type_atom_means = type_atom_stats[["type", "atom0", "atom1", "refined_mean"]]

type_atom0_stats = (
    train_merged.groupby(["type", "atom0"])["scalar_coupling_constant"]
    .agg(["mean", "count"])
    .reset_index()
)
type_atom0_stats["type_mean"] = type_atom0_stats["type"].map(type_means)
type_atom0_stats["refined_mean0"] = (
    type_atom0_stats["mean"] * type_atom0_stats["count"]
    + alpha * type_atom0_stats["type_mean"]
) / (type_atom0_stats["count"] + alpha)
type_atom0_means = type_atom0_stats[["type", "atom0", "refined_mean0"]]

type_atom1_stats = (
    train_merged.groupby(["type", "atom1"])["scalar_coupling_constant"]
    .agg(["mean", "count"])
    .reset_index()
)
type_atom1_stats["type_mean"] = type_atom1_stats["type"].map(type_means)
type_atom1_stats["refined_mean1"] = (
    type_atom1_stats["mean"] * type_atom1_stats["count"]
    + alpha * type_atom1_stats["type_mean"]
) / (type_atom1_stats["count"] + alpha)
type_atom1_means = type_atom1_stats[["type", "atom1", "refined_mean1"]]

print("Refined means for (type, atom0, atom1) computed. Sample:")
print(type_atom_means.head())



Input folders: ['mulliken_charges.csv.zip', 'magnetic_shielding_tensors.csv.zip', 'train.csv', 'potential_energy.csv.zip', 'structures.csv.zip', 'test.csv', 'structures', 'dipole_moments.csv.zip', 'test.csv.zip', 'structures.csv', 'description.md', 'sample_submission.csv', 'dipole_moments.csv', 'train.csv.zip', 'scalar_coupling_contributions.csv.zip', 'scalar_coupling_contributions.csv', 'magnetic_shielding_tensors.csv', 'structures.zip', 'potential_energy.csv', 'champs-scalar-coupling', 'sample_submission.csv.zip', 'mulliken_charges.csv']
Computed mean values for each type:
type
1JHC    94.971200
1JHN    47.519817
2JHC    -0.275012
2JHH   -10.284644
2JHN     3.122014
3JHC     3.689004
3JHH     4.773305
3JHN     0.989462
Name: scalar_coupling_constant, dtype: float64
Refined means for (type, atom0, atom1) computed. Sample:
   type atom0 atom1  refined_mean
0  1JHC     H     C     94.971200
1  1JHN     H     N     47.519817
2  2JHC     H     C     -0.275012
3  2JHH     H     H    -10.28

In [2]:
# ---------- Load and merge test data ----------
test_path = "../input/champs-scalar-coupling/test.csv"
test_df = pd.read_csv(test_path)

assert {"id", "type", "molecule_name", "atom_index_0", "atom_index_1"}.issubset(
    test_df.columns
), "Test file missing required columns."

test_merged = test_df.merge(
    structures_df[["molecule_name", "atom_index", "element"]].rename(
        columns={"atom_index": "atom_index_0", "element": "atom0"}
    ),
    on=["molecule_name", "atom_index_0"],
    how="left",
)
test_merged = test_merged.merge(
    structures_df[["molecule_name", "atom_index", "element"]].rename(
        columns={"atom_index": "atom_index_1", "element": "atom1"}
    ),
    on=["molecule_name", "atom_index_1"],
    how="left",
)

if test_merged[["atom0", "atom1"]].isnull().any().any():
    print(
        "Warning: some atom symbols missing in test merge; will fall back to type mean."
    )

test_pred = test_merged.merge(
    contrib_df[
        ["molecule_name", "atom_index_0", "atom_index_1", "type", "contrib_sum"]
    ],
    on=["molecule_name", "atom_index_0", "atom_index_1", "type"],
    how="left",
)

test_pred = test_pred.merge(type_atom_means, on=["type", "atom0", "atom1"], how="left")
test_pred = test_pred.merge(type_atom0_means, on=["type", "atom0"], how="left")
test_pred = test_pred.merge(type_atom1_means, on=["type", "atom1"], how="left")

global_type_mean = type_means.to_dict()
overall_global_mean = train_df["scalar_coupling_constant"].mean()

# ---------- Hierarchical fallback ----------
test_pred["scalar_coupling_constant"] = test_pred["contrib_sum"]

refined_cols = ["refined_mean", "refined_mean0", "refined_mean1"]
test_pred["avg_refined"] = test_pred[refined_cols].mean(axis=1, skipna=True)

test_pred["scalar_coupling_constant"] = test_pred["scalar_coupling_constant"].fillna(
    test_pred["avg_refined"]
)
test_pred["scalar_coupling_constant"] = test_pred["scalar_coupling_constant"].fillna(
    test_pred["type"].map(global_type_mean)
)
test_pred["scalar_coupling_constant"] = test_pred["scalar_coupling_constant"].fillna(
    overall_global_mean
)



In [3]:
submission_path = "submission.csv"
test_pred[["id", "scalar_coupling_constant"]].to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")

Submission written to submission.csv
